# 站站点数据与分布测试

本笔记本用于测试数据加载流程、验证项目配置，并可视化气象站的空间分布。

## 1. 环境设置

设置项目根目录，确保可以正常加载模块。

In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import folium

# 设置项目根目录
dir_path = '/Users/lxx/Documents/codes/st-missing-fill'
os.chdir(dir_path)
sys.path.append(dir_path)

print(f"当前工作目录: {os.getcwd()}")

# 加载配置
with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
    
print("配置加载成功。")

## 2. 数据加载

加载气象站的基础元数据信息。

In [ ]:
# 直接读取 Parquet 文件，不再依赖旧的 proc_data 模块
stations_file = 'data/meteo-swiss-description/stations.parquet'
print(f"读取站点文件: {stations_file}")
stations_info = pd.read_parquet(stations_file)
print(f"共加载 {len(stations_info)} 个站点信息。")
stations_info.head()

## 3. 空间分布可视化

使用 Folium 在地图上标注所有气象站的位置。

In [ ]:
# 确保保存目录存在
os.makedirs('data/figs', exist_ok=True)

# 创建地图，中心点设为所有站点的平均位置
m = folium.Map(
    location=[stations_info['latitude'].mean(), stations_info['longitude'].mean()],
    zoom_start=8,
    tiles='OpenStreetMap'
)

for idx, row in stations_info.iterrows():
    # 构建更丰富的弹出信息
    popup_text = f"""
    <b>名称:</b> {row['station_name']}<br>
    <b>高度:</b> {row['station_height']}m<br>
    <b>类型:</b> {row.get('station_type', 'N/A')}
    """
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        popup=folium.Popup(popup_text, max_width=300),
        color='#E74C3C',    # 边框颜色 (红色)
        weight=2,
        fill=True,
        fill_color='#3498DB', # 填充颜色 (蓝色)
        fill_opacity=0.7
    ).add_to(m)

# 保存地图
m.save('data/figs/stations_map.html')
print("地图已保存至: data/figs/stations_map.html")

# 在 Notebook 中显示地图
m